# Day 6: baseline results

What each condition got right, **why** it missed what it missed, what it cost, and where the north star stands.

Pick the benchmark and runs in the first cell. By default the notebook takes the latest run of each baseline for that benchmark from `eval/results/`.

| Condition | What the closed model sees |
|---|---|
| `muse_direct` | the full retrieved context (~13K tokens), no gateway |
| `escalate_compact` | the gateway always escalates, with the top-5 memory summaries |
| `rule_gate` | the gateway decides per turn (local or escalate) |
| `llama_only` | never escalates: the local model with memory |

**Outcome causes**, per item and condition:
- **hit:** right, with the evidence in view.
- **hit (guess):** right without the evidence, e.g. a path named in the issue.
- **model miss:** the evidence was in view, but the answer was wrong.
- **retrieval miss:** the evidence was in the full context, but not in the memory the gateway retrieved.
- **out of ceiling:** the evidence was never retrieved at all, so no condition could see it.

In [ ]:
BENCH = "swebench"   # or "locomo"
RUN_IDS = None       # e.g. ["20261028-101500", ...]; None = latest run per baseline
REPLAY_CSVS = None   # None = latest replay CSV per (baseline, order)

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from matplotlib.patches import Patch

RESULTS = Path("results") if Path("results").exists() else Path("eval/results")
ORDER = ["muse_direct", "escalate_compact", "rule_gate", "llama_only"]

# Validated colours (dataviz reference palette; all three pass all-pairs CVD checks).
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
SURFACE, INK, INK_2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
RAMP = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab"]  # ordinal blue steps (funnel stages)
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK_2, "xtick.color": MUTED, "ytick.color": MUTED,
    "text.color": INK, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.titlesize": 12,
    "axes.titleweight": "bold", "font.family": ["Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 110,
})


def load_summaries():
    rows = []
    for path in RESULTS.glob("*.json"):
        summary = json.loads(path.read_text())
        if "baseline" in summary:
            rows.append({**summary, "path": path})
    return pd.DataFrame(rows)


summaries = load_summaries()
if summaries.empty:
    raise SystemExit("No results yet: run `python -m eval.run ...` first.")
picked = summaries[summaries.bench == BENCH]
if RUN_IDS:
    picked = picked[picked.run_id.isin(RUN_IDS)]
else:
    picked = picked.sort_values("run_id").groupby("baseline").tail(1)
picked = picked.set_index("baseline").loc[[b for b in ORDER if b in set(picked.baseline)]]


def load_results(row):
    path = RESULTS / "raw" / f"{BENCH}_{row.name}_{row.run_id}.jsonl"
    return pd.DataFrame([json.loads(line) for line in path.read_text().splitlines()])


df = pd.concat([load_results(row) for _, row in picked.iterrows()], ignore_index=True)
df["correct"] = df.score.map(lambda s: bool(s.get("hit", s.get("correct"))))
conditions = list(picked.index)
items = sorted(set.intersection(*(set(df[df.baseline == b].item_id) for b in conditions)))
df = df[df.item_id.isin(items)].copy()
picked[["run_id", "items", "llama_model", "muse_model", "simulated", "muse_is_standin"]]

In [ ]:
if picked.simulated.any():
    source = "SIMULATED models: these numbers check the code, not the idea"
elif picked.muse_is_standin.any():
    source = "STAND-IN models (local Ollama): not results"
else:
    source = "REAL models"
display(Markdown(f"## {source}\n{len(items)} items common to {len(conditions)} conditions on **{BENCH}**."))

## Outcome of every item

Each row is one **pattern** of outcomes across the conditions, with the number of items that share it, most common first. Read across a row to see where the conditions disagree and why. The drill-down at the end shows single items.

In [ ]:
def cause(row):
    memory = row.baseline != "muse_direct"
    seen = bool(row.gold_in_hits) if memory else bool(row.in_context)
    if row.correct:
        return "hit" if seen else "hit (guess)"
    if seen:
        return "model miss"
    if memory and row.in_context:
        return "retrieval miss"
    return "out of ceiling"


df["cause"] = df.apply(cause, axis=1)
CAUSES = {  # colour, hatch, glyph
    "hit": (BLUE, None, "H"),
    "hit (guess)": (BLUE, "////", "g"),
    "model miss": (ORANGE, None, "M"),
    "retrieval miss": (AQUA, None, "R"),
    "out of ceiling": (SURFACE, None, "·"),
}
grid = df.pivot(index="item_id", columns="baseline", values="cause")[conditions]
patterns = grid.value_counts().reset_index(name="items")

fig, ax = plt.subplots(figsize=(1.6 * len(conditions) + 3.5, 0.32 * len(patterns) + 1.2))
for y, (_, row) in enumerate(patterns[conditions].iterrows()):
    for x, value in enumerate(row):
        colour, hatch, glyph = CAUSES[value]
        empty = value == "out of ceiling"
        ax.add_patch(plt.Rectangle((x, y), 1, 1, facecolor=colour, hatch=hatch,
                                   edgecolor=GRID if empty else SURFACE, linewidth=1 if empty else 2))
        ax.text(x + 0.5, y + 0.5, glyph, ha="center", va="center", fontsize=7,
                color=MUTED if empty else "white")
ax.set_xlim(0, len(conditions)); ax.set_ylim(len(patterns), 0)
ax.set_xticks(np.arange(len(conditions)) + 0.5, conditions); ax.xaxis.tick_top()
ax.set_yticks(np.arange(len(patterns)) + 0.5, [f"{n} items" for n in patterns["items"]], fontsize=8)
ax.grid(False); ax.tick_params(length=0)
for side in ("left", "bottom"):
    ax.spines[side].set_visible(False)
ax.legend(handles=[Patch(facecolor=c, hatch=h, edgecolor=GRID if k == "out of ceiling" else SURFACE,
                         label=f"{g}  {k}") for k, (c, h, g) in CAUSES.items()],
          loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False, title="Outcome",
          handleheight=1.2)
plt.tight_layout(); plt.show()

causes = df.groupby(["baseline", "cause"]).size().unstack(fill_value=0).reindex(conditions)
causes[[c for c in CAUSES if c in causes]]

## The north star: accuracy vs Muse-direct, and Muse tokens

**PASS** needs both of these:
- the 95% confidence interval of (condition − `muse_direct`) lies above **−3 points**, and
- the condition sends **≤ 10%** of Muse-direct's Muse tokens.

The intervals come from a paired bootstrap over items: each resample uses the same items for every condition.

In [ ]:
rng = np.random.default_rng(0)
correct = df.pivot(index="item_id", columns="baseline", values="correct")[conditions].astype(float)
B = 4000
idx = rng.integers(0, len(correct), size=(B, len(correct)))
boot = {c: correct[c].to_numpy()[idx].mean(axis=1) for c in conditions}

muse = df.assign(muse=df.muse_prompt_tokens + df.muse_completion_tokens).groupby("baseline").muse.sum()
rows = []
for c in conditions:
    acc = correct[c].mean() * 100
    lo, hi = np.percentile(boot[c] * 100, [2.5, 97.5])
    row = {"condition": c, "accuracy": acc, "acc_lo": lo, "acc_hi": hi}
    if "muse_direct" in conditions:
        delta = (boot[c] - boot["muse_direct"]) * 100
        row |= {"delta": acc - correct["muse_direct"].mean() * 100,
                "delta_lo": np.percentile(delta, 2.5), "delta_hi": np.percentile(delta, 97.5),
                "muse_token_share": muse.get(c, 0) / muse["muse_direct"] if muse["muse_direct"] else np.nan}
    rows.append(row)
table = pd.DataFrame(rows).set_index("condition")


def verdict(r):
    if r.name == "muse_direct" or "delta" not in r:
        return "reference"
    if r.muse_token_share > 0.10:
        return "FAIL: tokens over 10%"
    if r.delta_lo >= -3:
        return "PASS"
    if r.delta_hi < -3:
        return "FAIL: accuracy"
    return "not yet distinguishable"


table["verdict"] = table.apply(verdict, axis=1)
table.round(2)

In [ ]:
if "muse_direct" in conditions:
    others = [c for c in conditions if c != "muse_direct"]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 0.6 * len(others) + 1.8))
    ys = np.arange(len(others))
    t = table.loc[others]
    a1.axvspan(min(-3, t.delta_lo.min()) - 5, -3, color=GRID, alpha=0.6, lw=0)
    a1.axvline(0, color=AXIS, lw=1); a1.axvline(-3, color=MUTED, lw=1, ls="--")
    a1.errorbar(t.delta, ys, xerr=[t.delta - t.delta_lo, t.delta_hi - t.delta], fmt="o",
                color=BLUE, ecolor=BLUE, elinewidth=2, capsize=0, ms=8)
    for y, (c, r) in zip(ys, t.iterrows()):
        a1.annotate(f"{r.delta:+.1f} pts", (r.delta, y), xytext=(0, 9), textcoords="offset points",
                    ha="center", fontsize=9, color=INK_2)
    a1.set_yticks(ys, others); a1.invert_yaxis()
    a1.set_title("Accuracy vs muse_direct (points, 95% CI)", loc="left", pad=18)
    a1.text(-3, 1.01, "−3 pt limit", transform=a1.get_xaxis_transform(), color=MUTED,
            fontsize=8, ha="center", va="bottom")
    a1.grid(axis="y", visible=False)

    share = t.muse_token_share * 100
    a2.barh(ys, share, color=BLUE, height=0.5)
    a2.axvline(10, color=MUTED, lw=1, ls="--")
    for y, v in zip(ys, share):
        a2.text(v, y, f" {v:.1f}%", va="center", fontsize=9, color=INK_2)
    a2.set_yticks(ys, [""] * len(others)); a2.invert_yaxis()
    a2.set_xlim(0, max(12, share.max() * 1.25))
    a2.set_title("Muse tokens, % of muse_direct", loc="left", pad=18)
    a2.text(10, 1.01, "10% limit", transform=a2.get_xaxis_transform(), color=MUTED,
            fontsize=8, ha="center", va="bottom")
    a2.grid(axis="y", visible=False)
    plt.tight_layout(); plt.show()
    display(Markdown("\n".join(f"- **{c}**: {v}" for c, v in table.verdict.items())))

## Where items are lost

Each panel follows one condition from all items down to correct answers. The drop between two stages shows what lost those items: retrieval (the BM25 ceiling, then the gateway's top-5 memory) or the model.

In [ ]:
stages = ["all items", "in ceiling", "evidence shown", "correct"]
funnel = pd.DataFrame({
    c: [len(sub), sub.in_context.sum(),
        (sub.in_context if c == "muse_direct" else sub.gold_in_hits.fillna(False).astype(bool)).sum(),
        sub.correct.sum()]
    for c, sub in ((c, df[df.baseline == c]) for c in conditions)}, index=stages).T

fig, axes = plt.subplots(1, len(conditions), figsize=(3.2 * len(conditions), 2.6), sharex=True)
for ax, (c, row) in zip(np.atleast_1d(axes), funnel.iterrows()):
    ax.barh(stages, row.values, color=RAMP, height=0.6)
    for y, v in enumerate(row.values):
        ax.text(v, y, f" {v}", va="center", fontsize=9, color=INK_2)
    ax.invert_yaxis(); ax.set_title(c, loc="left", fontsize=10); ax.grid(axis="y", visible=False)
    if ax is not np.atleast_1d(axes)[0]:
        ax.set_yticks(range(len(stages)), [""] * len(stages))
plt.tight_layout(); plt.show()
funnel

## Cost vs accuracy

One point per condition. Up and to the left is better. Day 7's λ sweep turns this into a curve.

In [ ]:
per_item = df.assign(muse=df.muse_prompt_tokens + df.muse_completion_tokens).groupby("baseline").agg(
    muse=("muse", "mean"), accuracy=("correct", "mean")).reindex(conditions)
fig, ax = plt.subplots(figsize=(6.5, 4))
x = per_item.muse.clip(lower=1)
ax.scatter(x, per_item.accuracy * 100, s=70, color=BLUE, edgecolor=SURFACE, linewidth=2, zorder=3)
for c, r in per_item.iterrows():
    ax.annotate(c, (max(r.muse, 1), r.accuracy * 100), xytext=(7, 4), textcoords="offset points",
                fontsize=9, color=INK_2)
ax.set_xscale("log"); ax.set_xlabel("Muse tokens per item (log; 1 = none)")
ax.set_ylabel("accuracy (%)"); ax.set_title("Cost vs accuracy", loc="left")
plt.tight_layout(); plt.show()
per_item.round(3)

## Did the rule gate pick well?

For each item, the cheapest condition that gets it right is `llama_only` if it's right, else `escalate_compact` if that's right, else none. Comparing this with what `rule_gate` chose gives the **escalation regret**: the labels Day 7's learned gate trains on.

In [ ]:
if {"rule_gate", "llama_only", "escalate_compact"} <= set(conditions):
    w = correct[["llama_only", "escalate_compact"]].astype(bool)
    chosen = df[df.baseline == "rule_gate"].set_index("item_id").route.reindex(w.index)
    needed = np.where(w.llama_only, "local", np.where(w.escalate_compact, "escalate", "neither"))
    went = np.where(chosen.str.startswith("escalate"), "escalate", "local")
    labels = {("local", "local"): "right: stayed local",
              ("escalate", "escalate"): "right: escalated",
              ("escalate", "local"): "wasted: escalated, local was enough",
              ("local", "escalate"): "missed: stayed local, needed escalation"}
    regret = pd.Series([labels.get((g, n), f"{g}: nobody could") for g, n in zip(went, needed)],
                       index=w.index)
    counts = regret.value_counts()
    fig, ax = plt.subplots(figsize=(7.5, 0.45 * len(counts) + 1))
    ax.barh(counts.index, counts.values, color=BLUE, height=0.55)
    for y, v in enumerate(counts.values):
        ax.text(v, y, f" {v}", va="center", fontsize=9, color=INK_2)
    ax.invert_yaxis(); ax.grid(axis="y", visible=False)
    ax.set_title("rule_gate decisions vs the cheapest correct choice", loc="left")
    plt.tight_layout(); plt.show()
    display(counts.to_frame("items"))

## Cache: how much of each local prompt was already computed

One dot per local-model request in the order it was made. On vLLM and the simulator this is the cached share of prompt tokens; Ollama reports it too. Cache isolation between teams is checked by `tests/test_sim_e2e.py` and `scripts/cache_check.py`.

In [ ]:
local = df[df.llama_prompt_tokens > 0].reset_index(drop=True)
if len(local):
    share = local.cached_tokens / local.llama_prompt_tokens
    fig, ax = plt.subplots(figsize=(9, 2.8))
    ax.scatter(local.index, share * 100, s=16, color=BLUE, edgecolor=SURFACE, linewidth=0.5)
    ax.set_ylim(-3, 103); ax.set_xlabel("local request (in order)"); ax.set_ylabel("cached share (%)")
    ax.set_title(f"Prefix-cache reuse: {local.cached_tokens.sum() / local.llama_prompt_tokens.sum():.0%} "
                 "of local prompt tokens overall", loc="left")
    plt.tight_layout(); plt.show()

## Team scaling (replay)

Per-engineer cost as the team grows: each engineer always does the same number of tasks.
- **Shared** (blue): one team, with shared memory and cache.
- **No sharing** (orange): each engineer on their own.
- **Solid** lines are related tasks (a team working in one area); **dashed** lines are random tasks.

Compare blue with orange at the same line style. Related and random runs use different tasks, so their levels aren't comparable with each other.

In [ ]:
csvs = [Path(p) for p in REPLAY_CSVS] if REPLAY_CSVS else sorted(RESULTS.glob("replay_*.csv"))
if csvs:
    replay = pd.concat([pd.read_csv(p).assign(file=p.name) for p in csvs])
    replay = replay.sort_values("file").groupby(["baseline", "order", "team_size"]).tail(1)
    metrics = [("ingest_pages_per_user", "memory pages ingested / engineer"),
               ("compute_tokens_per_user", "local compute tokens / engineer"),
               ("muse_tokens_per_user", "Muse tokens / engineer")]
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
    for ax, (metric, title) in zip(axes, metrics):
        for (baseline, order), sub in replay.groupby(["baseline", "order"]):
            sub = sub.sort_values("team_size")
            colour = ORANGE if baseline == "no_sharing" else BLUE
            ax.plot(sub.team_size, sub[metric], color=colour, lw=2, marker="o", ms=5,
                    ls="--" if order == "random" else "-",
                    label=f"{'no sharing' if baseline == 'no_sharing' else 'shared'}, {order}")
        ax.set_xscale("log", base=2); ax.set_xticks(sorted(replay.team_size.unique()))
        ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
        ax.set_xlabel("team size"); ax.set_title(title, loc="left", fontsize=10)
        ax.set_ylim(bottom=0)
    axes[0].legend(frameon=False, fontsize=8)
    plt.tight_layout(); plt.show()
    display(replay[["baseline", "order", "team_size", "accuracy", "ingest_pages_per_user",
                    "compute_tokens_per_user", "muse_tokens_per_user", "cached_ratio"]])
else:
    print("No replay runs yet: python -m eval.run replay ...")

## Drill-down: one item

Set `ITEM` to any item id. The default is the first item where the conditions disagree.

In [ ]:
disagree = grid[grid.nunique(axis=1) > 1].index
ITEM = sorted(disagree)[0] if len(disagree) else grid.index[0]
rows = df[df.item_id == ITEM].set_index("baseline").loc[conditions]
print(f"{ITEM}   score per condition: " + ", ".join(f"{c}={r.correct}" for c, r in rows.iterrows()))
sources = next((s for s in rows.retrieved_sources if isinstance(s, list) and s), [])
print("memory the gateway retrieved:")
for s in sources:
    print("   ", s)
display(rows[["route", "cause", "correct", "in_context", "gold_in_hits", "muse_prompt_tokens",
              "llama_prompt_tokens", "cached_tokens", "answer"]])

## Export: escalation-regret labels for Day 7

One row per item: whether the local model (`llama_only`) and the compact escalation (`escalate_compact`) got it right. *Needs escalation* = only the escalation was right.

In [ ]:
if {"llama_only", "escalate_compact"} <= set(conditions):
    labels = correct[["llama_only", "escalate_compact"]].astype(bool).rename(
        columns={"llama_only": "local_correct", "escalate_compact": "escalated_correct"})
    labels["needs_escalation"] = ~labels.local_correct & labels.escalated_correct
    labels["source"] = source
    out = RESULTS / f"regret_labels_{BENCH}.csv"
    labels.to_csv(out)
    print(f"Wrote {out}: {labels.needs_escalation.sum()} of {len(labels)} items need escalation")